# 08 · Fine-Tuning an LLM with LoRA

**Time:** about 60 minutes (plus training) · **Prerequisites:** `07`

## Learning objectives
1. Explain LoRA: freeze a pretrained model and train small low-rank matrices
   added to its weights. With a 4-bit base model this is **QLoRA**.
2. Prepare chat-format data with **prompt masking**, so the loss covers only
   the answer.
3. **Measure the base model first**, on a held-out test set with confidence
   intervals. A strong base model leaves little to gain.
4. Find where fine-tuning *does* help: **baking instructions into the weights**
   so the prompt can be short.

### LoRA in one equation
A frozen weight matrix $W \in \mathbb{R}^{d \times k}$ gets a trainable update
of rank $r$:

$$ W' = W + \frac{\alpha}{r} BA, \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d, k) $$

For $d = k = 2048$ (this model's hidden size) and $r = 8$, $BA$ has $2 \cdot 2048 \cdot 8 = 32{,}768$
trainable numbers instead of $4.2$ million. $B$ starts at zero, so training
begins exactly at the pretrained model.

**Model.** We use `mlx-community/Qwen2.5-3B-Instruct-4bit`, a 3-billion-parameter
instruction-tuned model quantized to 4 bits (about 1.7 GB to download on first
use). Any MLX-LM chat model works: change `MODEL_NAME`, for example to
`mlx-community/Qwen2.5-0.5B-Instruct-4bit` on a machine with little memory.
Training takes several minutes on recent Apple Silicon; memory use depends on
batch size and sequence length.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import mlx.core as mx
from mlx_lm import generate, load
from mlx_lm.sample_utils import make_sampler

from mlx_nlp_utils import bootstrap_ci, print_device_info

print_device_info()
MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit"
DATA_DIR = Path("../data").resolve()
ADAPTER_DIR = Path("adapters").resolve()          # notebooks/adapters
greedy = make_sampler(temp=0.0)                    # deterministic decoding for evaluation

## 1. Load the pretrained model

In [ ]:
model, tokenizer = load(MODEL_NAME)
messages = [{"role": "user", "content": "What is the capital of France?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
print(generate(model, tokenizer, prompt=prompt, max_tokens=40, sampler=greedy))

`apply_chat_template` wraps messages in the special tokens this model was
trained with. Always use the tokenizer's own template; hand-written
delimiters from a different model family silently hurt quality.

## 2. The task and the data

The task: reply with exactly one label (`greeting`, `question`, or
`command`) for a user message, the intent task from notebook 01, now done
by an LLM. `data/train.jsonl` and `data/valid.jsonl` hold chat records:

In [ ]:
train_records = [json.loads(line) for line in (DATA_DIR / "train.jsonl").read_text().splitlines()]
valid_records = [json.loads(line) for line in (DATA_DIR / "valid.jsonl").read_text().splitlines()]
print(json.dumps(train_records[0], indent=1))

unique = lambda records: sorted({r["messages"][1]["content"] for r in records})
print(f"\ntrain: {len(train_records)} records, {len(unique(train_records))} unique prompts")
print(f"valid: {len(valid_records)} records, {len(unique(valid_records))} unique prompts: {unique(valid_records)}")

Look at those counts. The 800 training records are only ~37 distinct
prompts repeated, and the 200 validation records are only 8. The generator
splits prompts *before* repeating them, so nothing leaks, but **record count
is not example count**. The validation file is fine for watching the loss
curve; it is far too small to measure accuracy.

So we write a separate **test set**: new, hand-written messages that
appear in neither file and use different wording from the templates.

In [ ]:
SYSTEM = "Classify the user message. Reply with exactly one label: greeting, question, or command."
LABELS = ["greeting", "question", "command"]

test_set = [
    ("Good evening, nice to see you", "greeting"), ("Hiya!", "greeting"), ("Morning team", "greeting"),
    ("Hey hey, long time no see", "greeting"), ("Greetings from Berlin", "greeting"),
    ("Hello again", "greeting"), ("Sup", "greeting"), ("Hi, hope you're well", "greeting"),
    ("Is the pharmacy open on Sundays?", "question"), ("How far is the airport?", "question"),
    ("What's the exchange rate for yen?", "question"), ("Who wrote Pride and Prejudice?", "question"),
    ("Why does my laptop get so hot?", "question"), ("When is the next train?", "question"),
    ("Which planet is the largest?", "question"), ("Do you know a good pizza place?", "question"),
    ("Dim the bedroom lights", "command"), ("Add milk to my shopping list", "command"),
    ("Start a five minute timer", "command"), ("Call the dentist", "command"),
    ("Turn the heating up two degrees", "command"), ("Text Sarah that I'm running late", "command"),
    ("Shuffle my workout playlist", "command"), ("Book a taxi for 7am", "command"),
]
seen = set(unique(train_records)) | set(unique(valid_records))
assert not any(text in seen for text, _ in test_set), "test messages must be new"
print(f"{len(test_set)} test messages, {len(test_set) // 3} per label")

## 3. Measure the base model first

Without a baseline you can't tell whether fine-tuning helped. We score each
reply as correct only if its first word is the right label, and count
anything else (a sentence, a different word) as **invalid**, a failure
separate from choosing the wrong label.

We evaluate the base model two ways:

* **Instructed:** the system prompt spells out the task (what the training
  data contains).
* **Bare:** only the user's message, no instructions at all.

🤔 **Predict:** how accurate is the base model in each setting?

In [ ]:
def build_prompt(tokenizer, text, instructed):
    messages = ([{"role": "system", "content": SYSTEM}] if instructed else []) + [{"role": "user", "content": text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def classify(model, tokenizer, text, instructed):
    reply = generate(model, tokenizer, prompt=build_prompt(tokenizer, text, instructed),
                     max_tokens=6, sampler=greedy)
    words = reply.strip().lower().replace(".", " ").replace(":", " ").split()
    return (words[0] if words else ""), reply

def evaluate(model, tokenizer, name, instructed):
    rows = [(text, gold, *classify(model, tokenizer, text, instructed)) for text, gold in test_set]
    correct = [pred == gold for _, gold, pred, _ in rows]
    invalid = [pred not in LABELS for _, _, pred, _ in rows]
    acc, low, high = bootstrap_ci(correct)
    tokens = len(tokenizer.encode(build_prompt(tokenizer, test_set[0][0], instructed)))
    print(f"{name}: accuracy {acc:.0%} (95% CI {low:.0%}–{high:.0%}), invalid outputs {np.mean(invalid):.0%}, "
          f"prompt ≈ {tokens} tokens")
    return rows, acc

base_rows, base_acc = evaluate(model, tokenizer, "Base, instructed", instructed=True)
bare_rows, bare_acc = evaluate(model, tokenizer, "Base, bare      ", instructed=False)
print("\nBare base-model replies:")
for text, gold, pred, reply in bare_rows[:5]:
    print(f"   [{gold}] {text!r} → {reply.strip()!r}")

A capable instruction-tuned model, *given* instructions, already does this
three-way task well; there is little accuracy left for fine-tuning to add.
Check this first in any project: if prompting meets your bar, you may not need
to train anything.

Without instructions, though, the model has no idea it should answer with a
label, so it replies conversationally ("Hello! How can I assist…") and every
reply is scored invalid. (The scorer reads only the first six generated
tokens, which is plenty to see that a reply is not a label.) That points at
what fine-tuning can do here: make the model behave correctly **without**
the instruction in the request.

## 4. Fine-tune on bare prompts

We derive training data with the system message removed, so the model must
learn the task from examples alone. We call the `mlx_lm` LoRA trainer through
`sys.executable`, so it uses this notebook's environment. Settings worth understanding:

| Setting | Value | Why |
|---|---|---|
| `lora_parameters.rank` | 8 | Size of the low-rank update (the sweep in section 6 varies it) |
| `num_layers` | 16 | Add adapters to the last 16 of the model's 36 transformer blocks |
| `mask_prompt` | true | Loss only on the assistant's answer, not on the user text |
| `grad_checkpoint` | true | Recompute activations in the backward pass to save memory |
| `iters`, `batch_size` | 200, 2 | 400 sequences, about 10 passes over the ~37 unique training prompts |
| `learning_rate` | 1e-4 | A common LoRA starting point |

In [ ]:
import os
import re

BARE_DATA = ADAPTER_DIR / "data_bare"
BARE_DATA.mkdir(parents=True, exist_ok=True)
for name, records in [("train", train_records), ("valid", valid_records)]:
    with open(BARE_DATA / f"{name}.jsonl", "w") as f:
        for r in records:
            f.write(json.dumps({"messages": [m for m in r["messages"] if m["role"] != "system"]}) + "\n")

def train_adapter(adapter_dir, rank=8, iters=200, seed=0):
    """Write a config file and run the mlx_lm LoRA trainer quietly. Returns (seconds, log text)."""
    adapter_dir.mkdir(parents=True, exist_ok=True)
    config = {
        "model": MODEL_NAME, "train": True, "data": str(BARE_DATA), "adapter_path": str(adapter_dir),
        "fine_tune_type": "lora", "num_layers": 16, "batch_size": 2, "iters": iters,
        "learning_rate": 1e-4, "max_seq_length": 256, "grad_checkpoint": True, "mask_prompt": True,
        "lora_parameters": {"rank": rank, "dropout": 0.0, "scale": 20.0},
        "steps_per_report": 20, "steps_per_eval": 50, "val_batches": 25, "seed": seed,
    }
    config_path = adapter_dir / "lora_config.json"
    config_path.write_text(json.dumps(config, indent=1))    # JSON is valid YAML
    start = time.perf_counter()
    result = subprocess.run([sys.executable, "-m", "mlx_lm", "lora", "-c", str(config_path)],
                            capture_output=True, text=True, env={**os.environ, "TOKENIZERS_PARALLELISM": "false",
                                                                "NO_COLOR": "1", "TERM": "dumb"})
    log = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", result.stdout + result.stderr).replace("\r", "\n")
    (adapter_dir / "train.log").write_text(log)
    if result.returncode != 0:
        raise RuntimeError("Training failed; last log lines:\n" + "\n".join(log.strip().splitlines()[-15:]))
    return time.perf_counter() - start, log

def trainable_count(adapter_dir):
    return sum(w.size for w in mx.load(str(adapter_dir / "adapters.safetensors")).values())

seconds, log = train_adapter(ADAPTER_DIR, rank=8, iters=200)
print(next((l.strip() for l in log.splitlines() if "Trainable parameters" in l), ""))
val_losses = re.findall(r"^\s*(\d+)\s+val\s+([\d.]+)", log, flags=re.M)
print("validation loss by iteration:", ", ".join(f"{i}: {v}" for i, v in val_losses))
print(f"\nTrained in {seconds / 60:.1f} min; adapter file holds {trainable_count(ADAPTER_DIR):,} numbers "
      f"(the base model has about 3 billion)")

Validation loss is computed on only 8 distinct prompts, so treat it as a
sanity check that training worked, not as a quality estimate. The test set
below is the real measurement.

## 5. Evaluate the fine-tuned model

The fine-tuned model sees the same **bare** prompt that confused the base model.

In [ ]:
ft_model, ft_tokenizer = load(MODEL_NAME, adapter_path=str(ADAPTER_DIR))
ft_rows, ft_acc = evaluate(ft_model, ft_tokenizer, "Fine-tuned, bare", instructed=False)

print("\nComparison on the same 24 test messages:")
print(f"   {'':<20} {'accuracy':>9} {'prompt tokens':>14}")
for name, acc, instructed in [("Base, instructed", base_acc, True), ("Base, bare", bare_acc, False),
                              ("Fine-tuned, bare", ft_acc, False)]:
    n_tokens = len(tokenizer.encode(build_prompt(tokenizer, test_set[0][0], instructed)))
    print(f"   {name:<20} {acc:>9.0%} {n_tokens:>14}")

print("\nFine-tuned model errors:")
for text, gold, pred, reply in ft_rows:
    if pred != gold:
        print(f"   [{gold}] {text!r} → {reply.strip()!r}")

**Reading the result.** With 24 test messages each message moves accuracy by
about 4 points, and the confidence intervals are wide. What the table shows:

* The bare base model fails mostly through **format**: it does not know the
  expected output.
* The fine-tuned model, given no instructions, matches the instructed base
  model. Fine-tuning moved the instructions out of the prompt and into the
  weights.
* It did **not** beat the instructed base model. When a capable base model
  plus a prompt already meets your bar, fine-tuning does not buy accuracy.
* Look at the **prompt-token column**: the saving here is only a few tokens,
  because our instruction is one sentence. Fine-tuning pays off in prompt
  cost when the instruction (or a set of few-shot examples) is long and is
  sent with every request. For a one-sentence instruction, a prompt is
  simply cheaper than a training run.

Also look at the errors: items like "Do you know a good pizza place?" are
ambiguous (question or request?). Fine-tuning teaches *this dataset's*
convention, for better or worse.

## 6. Rank sweep (optional)

Does a larger rank help? Train ranks 2, 8, and 32 with the same budget and
compare on the test set. Set `RUN_RANK_SWEEP = True`; it trains three
adapters (`rank_sweep_iters` steps each), which takes several minutes.

In [ ]:
RUN_RANK_SWEEP = False
rank_sweep_iters = 100

if RUN_RANK_SWEEP:
    sweep = []
    for rank in [2, 8, 32]:
        path = ADAPTER_DIR / f"rank_{rank}"
        secs, _ = train_adapter(path, rank=rank, iters=rank_sweep_iters)
        m, t = load(MODEL_NAME, adapter_path=str(path))
        _, acc = evaluate(m, t, f"rank {rank}", instructed=False)
        sweep.append((rank, trainable_count(path), acc, secs))
        del m
    print(f"\n{'rank':>4} {'trainable':>11} {'test acc':>9} {'train min':>10}")
    for rank, params, acc, secs in sweep:
        print(f"{rank:>4} {params:>11,} {acc:>9.0%} {secs / 60:>10.1f}")
else:
    print("Skipped. Set RUN_RANK_SWEEP = True to train and compare three ranks.")

**What to expect.** On a three-label task, even rank 2 has far more capacity
than needed, so test accuracy is usually similar across ranks, within the
±4-points-per-message noise of a 24-message test set. Trainable parameters
grow linearly with rank. The general rule: increase rank only when a
held-out metric shows the smaller rank is underfitting. One run per rank is
an anecdote; a real sweep uses several seeds and a larger test set.

## Summary
* LoRA trains a low-rank update $BA$ on top of frozen weights; QLoRA does
  this over a 4-bit base model, so a 3B model fine-tunes on a laptop.
* Prompt masking puts the loss on the answer only.
* **Measure the base model first**, on held-out data with different wording,
  and report intervals. Count invalid outputs separately from wrong labels.
* A strong instruct model plus a good prompt may already meet your bar. Fine-tuning
  earns its place when instructions or examples are long enough that removing
  them from every request saves real cost, or when you want behavior you can't
  describe in words.

## 🧠 Exercises

**1. Instructed fine-tuned model.** Evaluate `ft_model` *with* the system
prompt (`instructed=True`). 🤔 Predict: better, worse, or the same as bare?

<details><summary>What to look for</summary>

The model was trained without a system message, so adding one is a small
distribution shift. It usually still works, but it is no longer the setting it
was trained on. Always evaluate in the exact format you will deploy.
</details>

**2. Few-shot baseline.** Give the *bare base model* three labeled examples in
the user message (one per label) and score it. 🤔 Predict: how much of the
fine-tuning gain does a few-shot prompt recover?

<details><summary>What to look for</summary>

Often most of it, at the price of a longer prompt on every request. Compare
accuracy *and* prompt tokens: that is the real trade-off between few-shot
prompting and fine-tuning.
</details>

**3. Catastrophic forgetting.** Ask the fine-tuned model "What is the capital
of France?" 🤔 What happens?

<details><summary>What to look for</summary>

A model fine-tuned to answer with one of three labels may now answer with a
label even for unrelated questions. Fine-tuning on a narrow task can override general
behavior. LoRA makes this reversible: load the model without the adapter and
the base model is unchanged.
</details>

**4. Data efficiency.** Regenerate the data with `python ../scripts/generate_synthetic_data.py --lora-train 100`
and retrain. Does accuracy drop?

## ❓ FAQ

**How is this different from full fine-tuning?** Full fine-tuning updates
all weights and needs memory for their gradients and optimizer state. LoRA
updates only the small adapter matrices.

**What does "4-bit" mean?** The base weights are stored in 4 bits (plus
scaling metadata). It reduces weight memory; activations, adapters, and
optimizer state still use more precision.

**Next:** `09_RAG_from_Scratch` gives a model new knowledge *without*
training, by retrieving it at question time.